# Create .npz for regions

# Create one .npz

In [2]:
import json
import itertools
from pathlib import Path

import numpy as np


# =========================
# CONFIG
# =========================
INPUTS = [
    "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Ladzany.npz",
    "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Sasovske.npz",
    "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Sitno.npz",
]

GAP = 64
DEFAULT_FILL_VALUE = -255.0
SQUEEZE_SINGLE_CHANNEL = False


# =========================
# HELPERS
# =========================
def parse_meta(meta_raw):
    if meta_raw is None:
        return {}

    if isinstance(meta_raw, bytes):
        meta_raw = meta_raw.decode("utf-8")

    if isinstance(meta_raw, np.ndarray):
        if meta_raw.shape == ():
            meta_raw = meta_raw.item()
        else:
            return {}

    if isinstance(meta_raw, str):
        try:
            return json.loads(meta_raw)
        except Exception:
            return {}

    if isinstance(meta_raw, dict):
        return meta_raw

    return {}


def to_chw(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)

    if arr.ndim == 2:
        return arr[None, ...]

    if arr.ndim >= 3:
        # assume CHW if first dim looks like channels
        if arr.shape[0] <= 8:
            return arr
        # otherwise try HWC -> CHW
        if arr.shape[-1] <= 8:
            return np.transpose(arr, (2, 0, 1))

    raise ValueError(f"Unsupported data shape: {arr.shape}")


def load_npz(path: str):
    z = np.load(path, allow_pickle=True)

    data = to_chw(z["data"]).astype(np.float32)
    mask = z["mask"].astype(np.uint8)
    valid = z["valid"].astype(np.uint8)

    if mask.shape != valid.shape:
        raise ValueError(f"{path}: mask.shape != valid.shape -> {mask.shape} vs {valid.shape}")

    if data.shape[1:] != mask.shape:
        raise ValueError(f"{path}: data spatial shape != mask.shape -> {data.shape[1:]} vs {mask.shape}")

    meta = parse_meta(z["meta"]) if "meta" in z.files else {}
    fill_value = float(DEFAULT_FILL_VALUE)

    return {
        "path": path,
        "name": Path(path).stem,
        "data": data,
        "mask": mask,
        "valid": valid,
        "meta": meta,
        "fill_value": fill_value,
        "C": data.shape[0],
        "H": data.shape[1],
        "W": data.shape[2],
    }


def split_rows(order, break_mask):
    rows = []
    current = [order[0]]

    for i in range(len(order) - 1):
        if (break_mask >> i) & 1:
            rows.append(current)
            current = [order[i + 1]]
        else:
            current.append(order[i + 1])

    rows.append(current)
    return rows


def evaluate_layout(sources, rows, gap):
    row_dims = []
    for row in rows:
        widths = [sources[i]["W"] for i in row]
        heights = [sources[i]["H"] for i in row]

        row_w = sum(widths) + gap * (len(row) - 1)
        row_h = max(heights)

        row_dims.append((row_w, row_h))

    canvas_w = max(w for w, _ in row_dims)
    canvas_h = sum(h for _, h in row_dims) + gap * (len(row_dims) - 1)
    area = canvas_w * canvas_h

    return area, canvas_w, canvas_h, row_dims


def find_best_layout(sources, gap=64):
    n = len(sources)
    best = None

    for perm in itertools.permutations(range(n)):
        for break_mask in range(1 << (n - 1)):
            rows = split_rows(perm, break_mask)
            area, canvas_w, canvas_h, row_dims = evaluate_layout(sources, rows, gap)

            candidate = {
                "area": area,
                "canvas_w": canvas_w,
                "canvas_h": canvas_h,
                "rows": rows,
                "row_dims": row_dims,
            }

            if best is None:
                best = candidate
                continue

            # minimize area, then height, then width
            if (
                candidate["area"] < best["area"]
                or (
                    candidate["area"] == best["area"]
                    and candidate["canvas_h"] < best["canvas_h"]
                )
                or (
                    candidate["area"] == best["area"]
                    and candidate["canvas_h"] == best["canvas_h"]
                    and candidate["canvas_w"] < best["canvas_w"]
                )
            ):
                best = candidate

    return best


def merge_sources(sources, gap=64, squeeze_single_channel=True):
    if not sources:
        raise ValueError("No sources provided")

    channel_counts = {src["C"] for src in sources}
    if len(channel_counts) != 1:
        raise ValueError(f"Different channel counts found: {channel_counts}")

    fill_values = {src["fill_value"] for src in sources}
    if len(fill_values) != 1:
        print(f"[WARN] Different fill_value found: {fill_values}. Using first one.")

    fill_value = sources[0]["fill_value"]
    C = sources[0]["C"]

    best = find_best_layout(sources, gap=gap)

    H = best["canvas_h"]
    W = best["canvas_w"]

    merged_data = np.full((C, H, W), fill_value, dtype=np.float32)
    merged_mask = np.zeros((H, W), dtype=np.uint8)
    merged_valid = np.zeros((H, W), dtype=np.uint8)

    placements = []

    y0 = 0
    for row, (_, row_h) in zip(best["rows"], best["row_dims"]):
        x0 = 0
        for idx in row:
            src = sources[idx]
            h, w = src["H"], src["W"]

            merged_data[:, y0:y0+h, x0:x0+w] = src["data"]
            merged_mask[y0:y0+h, x0:x0+w] = src["mask"]
            merged_valid[y0:y0+h, x0:x0+w] = src["valid"]

            # force invalid pixels to fill_value
            merged_data[:, y0:y0+h, x0:x0+w][:, src["valid"] == 0] = fill_value
            merged_mask[y0:y0+h, x0:x0+w][src["valid"] == 0] = 0

            placements.append({
                "name": src["name"],
                "path": src["path"],
                "y": int(y0),
                "x": int(x0),
                "height": int(h),
                "width": int(w),
            })

            x0 += w + gap

        y0 += row_h + gap

    meta = {
        "type": "merged_baseline_canvas",
        "fill_value": fill_value,
        "gap": int(gap),
        "channels": int(C),
        "height": int(H),
        "width": int(W),
        "placements": placements,
    }

    if squeeze_single_channel and C == 1:
        merged_data = merged_data[0]

    return merged_data, merged_mask, merged_valid, meta


# =========================
# RUN
# =========================
sources = [load_npz(p) for p in INPUTS]

merged_data, merged_mask, merged_valid, merged_meta = merge_sources(
    sources,
    gap=GAP,
    squeeze_single_channel=SQUEEZE_SINGLE_CHANNEL,
)

to_save = {
    "data": np.float32,   # [H, W]
    "mask": np.uint8,     # [H, W]
    "valid": np.uint8,    # [H, W]
    "meta": None
}

to_save["data"] = merged_data.astype(np.float32)
to_save["mask"] = merged_mask.astype(np.uint8)
to_save["valid"] = merged_valid.astype(np.uint8)
to_save["meta"] = json.dumps(merged_meta)

print(f"data shape: {merged_data.shape}")
print(f"mask shape: {merged_mask.shape}")
print(f"valid shape: {merged_valid.shape}")
print(json.dumps(merged_meta, indent=2))

data shape: (7, 9533, 6139)
mask shape: (9533, 6139)
valid shape: (9533, 6139)
{
  "type": "merged_baseline_canvas",
  "fill_value": -255.0,
  "gap": 64,
  "channels": 7,
  "height": 9533,
  "width": 6139,
  "placements": [
    {
      "name": "Ladzany",
      "path": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Ladzany.npz",
      "y": 0,
      "x": 0,
      "height": 5411,
      "width": 2895
    },
    {
      "name": "Sasovske",
      "path": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Sasovske.npz",
      "y": 0,
      "x": 2959,
      "height": 2799,
      "width": 3180
    },
    {
      "name": "Sitno",
      "path": "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/thesis/notebooks/packed/Sitno.npz",
      "y": 5475,
      "x": 0,
      "height": 4058,
      "width": 5480
    }
  ]
}


In [3]:
np.savez_compressed(
    "../data/processed/DEM(sk)_raw.npz",
    **to_save
)

Saved: ../data/processed/DEM(sk)_raw.npz
